# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why
This task is a binary classification problem: predicting whether a piece of content represents a refresh opportunity.

I selected Gradient Boosting as the final modeling approach because it can capture complex nonlinear relationships between content performance signals and refresh opportunities. Compared with a single Decision Tree, boosting combines multiple weak learners to improve predictive performance and reduce bias.

The model is appropriate for this lane because content refresh decisions depend on interactions between several signals such as traffic, engagement, content age, and historical performance.

The primary evaluation metric is Recall because missing a real refresh opportunity (false negative) has a higher business cost than reviewing some additional content that may not require an update.

The model is compared against the Week-4 baseline using the same dataset split and evaluation metrics.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

url = "https://raw.githubusercontent.com/estuchis21/ml-internship-week1/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df.columns


df["refresh_flag"] = (
    (df["trend_pct"] < -0.20) &
    (df["avg_position"] > 5)
).astype(int)


df["refresh_flag"].value_counts()

y = df["refresh_flag"]

numeric_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns




features = [
    "days_with_impressions",
    "word_count",
    "char_count",
    "days_since_last_update",
    "days_with_sessions",
    "ai_sessions_90d",
    "pageviews_90d",
    "users_90d",
    "sessions_90d",
    "clicks_90d",
    "impressions_90d",
    "clicks_last_30d",
    "impressions_last_30d",
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]


numeric_features = df[features].select_dtypes(
    include=["int64","float64"]
).columns


correlation_features = (
    df[list(numeric_features) + ["refresh_flag"]]
    .corr()["refresh_flag"]
    .drop("refresh_flag")
    .sort_values(
        ascending=False
    )
)


X = df[features]

y = df["refresh_flag"]

correlation_features

,refresh_flag
days_with_impressions,0.292108
word_count,0.185375
char_count,0.154016
days_since_last_update,0.088517
days_with_sessions,0.037737
ai_sessions_90d,0.034539
pageviews_90d,0.031494
users_90d,0.024626
sessions_90d,0.022475
scroll_events_90d,0.021370


## 2. Split design

The dataset was divided into training and testing sets using a stratified random split.

A stratified split was selected because the target variable is imbalanced, and maintaining the same proportion of refresh opportunities in both datasets ensures a fair evaluation.

The test set was kept completely separate from training to measure how well the model generalizes to unseen content.

The split configuration:

- Training set: 80%
- Test set: 20%
- Random state: 42
- Stratification based on the target variable

This approach avoids evaluating the model on data it has already seen and provides an honest estimate of performance.

In [2]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

(24000, 25)
(6000, 25)
refresh_flag
1    13698
0    10302
Name: count, dtype: int64
refresh_flag
1    3425
0    2575
Name: count, dtype: int64


## 3. Train + compare vs baseline

The Gradient Boosting model was trained using the same dataset split as the Week-4 baseline to ensure a fair comparison.

The model was evaluated using the same metrics, with Recall as the primary metric because the main objective is identifying as many potential content refresh opportunities as possible.

The model outputs probabilities for each content item, which allows ranking pages by their likelihood of requiring a refresh instead of only making binary predictions.

The final evaluation compares the baseline performance against the Gradient Boosting model to measure the improvement obtained from the new approach.

The Week-4 baseline used a simple rule-based scoring approach based on content age and recent impressions. While this provided an interpretable prioritization strategy, it relied on only two signals and fixed thresholds.

The final Gradient Boosting model was trained using multiple content performance features and evaluated on the same train/test split. Compared with the baseline, the machine learning model achieved higher measured recall and overall discrimination on the held-out test set.

This observed improvement suggests that combining multiple content signals allows the model to identify refresh opportunities more effectively than the rule-based baseline. The Gradient Boosting model was therefore selected as the final decision-support model for this project.

In [3]:
# ============================
# TRY DIFFERENT RANDOM FORESTS
# ============================
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.inspection import permutation_importance
rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight={
        0:1,
        1:2
    },
    random_state=42,
    n_jobs=-1
)


rf.fit(
    X_train,
    y_train
)


from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier


gb = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "model",
        GradientBoostingClassifier(
            n_estimators=500,
            learning_rate=0.005,
            max_depth=6,
            random_state=42
        )
    )

])


gb.fit(
    X_train,
    y_train
)

y_pred = gb.predict(
    X_test
)


y_prob = gb.predict_proba(
    X_test
)[:,1]


## 4. Errors and interpretation

The model errors were analyzed using false positives and false negatives.

False negatives are the most important errors because they represent content that could benefit from a refresh but was not identified by the model.

Feature importance was inspected to understand which signals influenced the predictions.

The goal is not only to improve metrics, but also to create an interpretable ranking of content refresh opportunities.

In [7]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


y_pred = gb.predict(X_test)

y_prob = gb.predict_proba(X_test)[:,1]


results_scores = pd.DataFrame({

    "Model": [
        "Gradient Boosting"
    ],

    "Accuracy": [
        accuracy_score(
            y_test,
            y_pred
        )
    ],

    "Precision": [
        precision_score(
            y_test,
            y_pred
        )
    ],

    "Recall": [
        recall_score(
            y_test,
            y_pred
        )
    ],

    "F1": [
        f1_score(
            y_test,
            y_pred
        )
    ],

    "ROC-AUC": [
        roc_auc_score(
            y_test,
            y_prob
        )
    ]
})

print(results_scores)

results = X_test.copy()

results["Actual"] = y_test.values
results["Predicted"] = y_pred

false_negatives = results[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0)
]

print(false_negatives.head(10))

false_positives = results[
    (results["Actual"] == 0) &
    (results["Predicted"] == 1)
]

print(false_positives.head(10))

results

               Model  Accuracy  Precision    Recall        F1   ROC-AUC
0  Gradient Boosting  0.859333   0.819827  0.965839  0.886863  0.922603
       days_with_impressions  word_count  char_count  days_since_last_update  \
29033                     88         NaN         NaN                      22   
5842                      88      2990.0     18321.0                     104   
28858                     88         NaN         NaN                      35   
14797                     88         NaN         NaN                     104   
3559                      88         NaN         NaN                      20   
20789                     88      2817.0     16903.0                      26   
29800                     88         NaN         NaN                     104   
7124                      88      2709.0     16469.0                      14   
410                       88         NaN         NaN                     104   
22402                     88         NaN         NaN    

,days_with_impressions,word_count,char_count,days_since_last_update,days_with_sessions,ai_sessions_90d,pageviews_90d,users_90d,sessions_90d,clicks_90d,...,ctr,engagement_rate,scroll_rate,engaged_sessions_90d,scroll_events_90d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,Actual,Predicted
27832,72,1814.0,11444.0,98,4,0,4,4,4,0,...,0.00,0.00,75.00,0,3,43,0,3,0,0
13753,88,NaN,NaN,104,51,0,208,198,199,42,...,0.22,2.51,6.73,5,14,8283,17,127,1,1
3666,6,1258.0,8534.0,20,1,0,1,1,1,0,...,0.00,0.00,0.00,0,0,4,0,0,1,1
4521,88,5413.0,34112.0,104,34,0,193,158,158,5,...,0.05,0.00,7.77,0,15,2668,1,11,1,1
5476,88,NaN,NaN,22,10,1,11,10,11,4,...,0.45,9.09,81.82,1,9,198,0,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26898,86,2996.0,19442.0,20,17,4,38,29,31,9,...,0.12,0.00,57.89,0,22,3524,4,13,1,1
20994,83,3102.0,19841.0,7,3,0,4,4,4,0,...,0.00,0.00,0.00,0,0,180,0,3,1,1
8145,88,NaN,NaN,104,5,0,6,5,5,11,...,0.26,0.00,16.67,0,1,1660,2,2,0,1
25590,88,3621.0,25418.0,20,78,0,1638,1337,1444,1347,...,4.78,2.42,43.22,35,708,7537,325,333,1,1


## Model evaluation results

The Gradient Boosting model achieved strong performance on the test set.

The model reached a recall of 97%, meaning it successfully identifies the majority of content refresh opportunities. This metric was prioritized because false negatives represent missed opportunities for improving content performance.

The model also achieved a precision of 82%, showing that most predicted refresh opportunities are relevant.

The ROC-AUC score of 82.8% indicates that the model can effectively distinguish between content that requires refreshing and content that does not.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.